# Fine-Tuning in klein — ein Sprachmodell lernt Stimmung erkennen

**Der Gedanke in einem Satz:** Wir nehmen ein Modell, das schon *sehr viel*
Text gesehen hat (es „versteht" Sprache grob), und trainieren es mit **ein paar
wenigen eigenen Beispielen** kurz nach — danach kann es *unsere* Aufgabe.

Unsere Aufgabe: Sätze als **positiv** oder **negativ** einsortieren (Sentiment).

Wir machen drei Schritte:
1. Ein vortrainiertes Mini-Sprachmodell laden (`distilbert-base-uncased`).
2. **Vorher:** prüfen, dass es unsere Aufgabe noch *nicht* kann (rät nur).
3. **Fine-Tuning:** mit ~16 Beispielsätzen kurz nachtrainieren — und sehen, dass
   es die Aufgabe danach kann, auch bei neuen Sätzen.

> Genau dieser Gedanke steckt auch hinter dem Finetunen großer Foundation-
> Modelle (Sprache, Bilder, Zeitreihen) — nur hier in winzig und in Sekunden.

## 1. Vortrainiertes Modell laden

`distilbert-base-uncased` wurde auf riesigen Textmengen vortrainiert. Der
**Körper** (das Sprachverständnis) ist also schon gelernt; nur der **Klassifikations-
Kopf** für „positiv/negativ" ist neu und noch zufällig — den bringen wir gleich bei.

> Beim Laden meldet `transformers`, dass die `classifier`-Gewichte neu
> initialisiert wurden — genau das ist gemeint: der Kopf ist noch untrainiert.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()   # Lade-Hinweise leiser stellen

DEVICE = "mps" if torch.backends.mps.is_available() else (
         "cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "models/distilbert-base-uncased"   # lokal, sonst Hugging-Face-ID
import os
if not os.path.isdir(MODEL_ID):
    MODEL_ID = "distilbert-base-uncased"
print("Device:", DEVICE, "| Modell:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2,        # 2 Klassen: 0=negativ, 1=positiv
).to(DEVICE)
print("Modell geladen — der Klassifikations-Kopf ist noch untrainiert.")

Device: mps | Modell: models/distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Modell geladen — der Klassifikations-Kopf ist noch untrainiert.


## 2. Unsere kleinen Daten

16 Trainingssätze (8 positiv, 8 negativ) und ein paar **neue** Testsätze, die das
Modell beim Training nie sieht. So winzig darf es sein — es geht um den Gedanken.

In [2]:
train_texts = [
    "I love this movie, it was fantastic",
    "What a wonderful and delightful experience",
    "Absolutely brilliant, I really enjoyed it",
    "This is the best thing I have ever bought",
    "The food was delicious and the staff was friendly",
    "A truly amazing and inspiring story",
    "I am so happy with this product",
    "Great quality and excellent service",
    "I hate this movie, it was terrible",
    "What an awful and disappointing experience",
    "Absolutely horrible, I really regret it",
    "This is the worst thing I have ever bought",
    "The food was disgusting and the staff was rude",
    "A truly boring and depressing story",
    "I am so unhappy with this product",
    "Poor quality and terrible service",
]
train_labels = [1]*8 + [0]*8     # 1 = positiv, 0 = negativ

test_texts = [
    "I really enjoyed every minute of it",      # positiv
    "This was a great and pleasant surprise",   # positiv
    "It was a complete waste of money",         # negativ
    "I would never recommend this to anyone",   # negativ
]
print(f"{len(train_texts)} Trainingssätze, {len(test_texts)} neue Testsätze")

16 Trainingssätze, 4 neue Testsätze


## 3. Vorher: Kann das Modell die Aufgabe schon?

Wir lassen das frisch geladene Modell die Testsätze einschätzen. Da der Kopf
noch untrainiert ist, sind die Antworten quasi **geraten**.

In [3]:
@torch.no_grad()
def predict(texts):
    model.eval()
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)
    preds = probs.argmax(dim=-1)
    return preds.cpu().tolist(), probs.cpu()

label_name = {1: "positiv", 0: "negativ"}

def show(texts):
    preds, probs = predict(texts)
    for t, p, pr in zip(texts, preds, probs):
        print(f"  [{label_name[p]:7}] (p={pr[p]:.2f})  {t}")

print("VORHER (nur geladen, nicht fine-getunt):")
show(test_texts)

VORHER (nur geladen, nicht fine-getunt):
  [positiv] (p=0.54)  I really enjoyed every minute of it
  [positiv] (p=0.53)  This was a great and pleasant surprise
  [positiv] (p=0.52)  It was a complete waste of money
  [positiv] (p=0.53)  I would never recommend this to anyone


## 4. Fine-Tuning — der eigentliche Schritt

Das ist ein ganz normaler kleiner Trainings-Loop: Sätze in Zahlen umwandeln
(tokenisieren), Modell vorhersagen lassen, Fehler (Loss) berechnen, Gewichte
ein bisschen anpassen — ein paar Mal über die 16 Sätze (Epochen).

In [4]:
from torch.optim import AdamW

# Daten einmal tokenisieren
enc = tokenizer(train_texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
labels = torch.tensor(train_labels).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(1, 11):                      # 10 Epochen reichen bei so wenig Daten
    optimizer.zero_grad()
    out = model(**enc, labels=labels)           # Modell berechnet den Loss selbst
    out.loss.backward()                          # Gradienten
    optimizer.step()                             # Gewichte anpassen
    print(f"Epoche {epoch:2d}  Loss = {out.loss.item():.4f}")

print("\nFine-Tuning fertig.")

Epoche  1  Loss = 0.6838
Epoche  2  Loss = 0.6718
Epoche  3  Loss = 0.6090
Epoche  4  Loss = 0.5499
Epoche  5  Loss = 0.4558
Epoche  6  Loss = 0.3771
Epoche  7  Loss = 0.2964
Epoche  8  Loss = 0.2304


Epoche  9  Loss = 0.1723
Epoche 10  Loss = 0.1301

Fine-Tuning fertig.


## 5. Nachher: dieselben neuen Sätze

Dieselben Testsätze wie oben — jetzt nach dem Fine-Tuning.

In [5]:
print("NACHHER (fine-getunt):")
show(test_texts)

NACHHER (fine-getunt):
  [positiv] (p=0.90)  I really enjoyed every minute of it
  [positiv] (p=0.91)  This was a great and pleasant surprise
  [negativ] (p=0.89)  It was a complete waste of money
  [negativ] (p=0.67)  I would never recommend this to anyone


## 6. Das war's — der ganze Gedanke

- **Vortrainiert** = das Modell hatte schon ein grobes Sprachverständnis.
- **Fine-Tuning** = ein kurzer Nachschliff auf *unseren* paar Beispielen.
- Ergebnis: Mit **16 Sätzen** und **10 Epochen** (Sekunden) lernt es eine Aufgabe,
  die es vorher nicht konnte — und überträgt das auf **neue** Sätze.

Das ist exakt dasselbe Prinzip wie beim Finetunen großer Foundation-Modelle
(GPT/BERT für Text, Bildmodelle, Zeitreihenmodelle wie TimesFM) — nur dass die
„groß"-Variante mehr Daten, mehr Rechenzeit und Tricks wie LoRA braucht, damit
das Nachtrainieren bezahlbar bleibt.

**Selbst ausprobieren:** eigene Sätze in `train_texts`/`test_texts` schreiben,
`lr` oder die Epochenzahl ändern.